# 05 — Tải checkpoint, đánh giá lại và backtest

Notebook độc lập, không train hoặc tải lại Yahoo. Add Input **toàn bộ output run** của notebook 03/04 rồi đặt `RUN_PATH`. Dữ liệu test và scaler đã lưu bảo đảm đánh giá đúng tập cũ.

## 1. Chọn thư mục run

In [ ]:
from pathlib import Path

RUN_PATH = ""  # Ví dụ /kaggle/input/my-training-output/hci/GSPC_comparison_.../
CONFIG = {"SOURCE": "csv", "RUN_WAVELET": False}
EVAL_OUTPUT_ROOT = "/kaggle/working/hci_evaluation" if Path("/kaggle/working").exists() else "kaggle_evaluation"
COMMISSION = 0.001


In [ ]:
import importlib.util
import subprocess
import sys

# Dùng PyTorch/CUDA có sẵn của Kaggle. Chỉ cài gói phụ nếu thiếu.
required = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "torch": "torch"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f"Thiếu thư viện nền: {missing}. Chọn image Python/GPU Kaggle hoặc cài môi trường local trước.")
optional = {}
if CONFIG.get("SOURCE") == "yfinance":
    optional["yfinance"] = "yfinance>=0.2.40"
if CONFIG.get("SOURCE") == "kaggle":
    optional["kagglehub"] = "kagglehub>=0.3.0"
if CONFIG.get("RUN_WAVELET"):
    optional["pywt"] = "PyWavelets>=1.4"
for module, package in optional.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])


## Tiện ích

In [ ]:
import hashlib
import importlib.metadata
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def select_device(require_gpu=False):
    if require_gpu and not torch.cuda.is_available():
        raise RuntimeError("Chưa có GPU. Bật Accelerator GPU trên Kaggle rồi chạy lại; hoặc đặt REQUIRE_GPU=False để thử CPU.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
    return device


def new_run(config, label):
    base = Path(config["OUTPUT_ROOT"])
    base.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    symbol = re.sub(r"[^A-Za-z0-9._-]", "_", config['TICKER'].replace('^', ''))
    root = base / f"{symbol}_{label}_{stamp}"
    root.mkdir(exist_ok=False)
    save_json(root / "config.json", config)
    versions = {}
    for name in ["numpy", "pandas", "scikit-learn", "torch", "matplotlib", "yfinance", "PyWavelets", "kagglehub"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    save_json(root / "environment.json", {"packages": versions, "cuda_runtime": torch.version.cuda,
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
    print("Output:", root)
    return root


## Đọc scaler đã lưu
Không fit lại scaler và không thay đổi split.

In [ ]:
def add_features(raw):
    # Tất cả phép tính chỉ dùng giá hiện tại/quá khứ; không backward fill.
    df = raw.copy()
    close = df.Close
    df["SMA_10"] = close.rolling(10).mean()
    df["SMA_20"] = close.rolling(20).mean()
    df["EMA_12"] = close.ewm(span=12, adjust=False).mean()
    df["EMA_26"] = close.ewm(span=26, adjust=False).mean()
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    df["RSI_14"] = 100 - 100 / (1 + gain / (loss + 1e-9))
    df["MACD"] = df.EMA_12 - df.EMA_26
    df["MACD_signal"] = df.MACD.ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df.MACD - df.MACD_signal
    std = close.rolling(20).std()
    df["BB_upper"] = df.SMA_20 + 2 * std
    df["BB_lower"] = df.SMA_20 - 2 * std
    df["BB_width"] = (df.BB_upper - df.BB_lower) / df.SMA_20
    tr = pd.concat([df.High - df.Low, (df.High - close.shift()).abs(),
                    (df.Low - close.shift()).abs()], axis=1).max(axis=1)
    df["ATR_14"] = tr.rolling(14).mean()
    df["OBV"] = (np.sign(close.diff()).fillna(0) * df.Volume).cumsum()
    df = df.dropna().copy()
    if not np.isfinite(df.to_numpy()).all():
        raise ValueError("Đặc trưng chứa NaN/Inf.")
    return df


def scaler_state(scaler):
    return {name: getattr(scaler, name).tolist() for name in
            ["scale_", "min_", "data_min_", "data_max_", "data_range_"]}


def inverse_target(values, meta):
    s = meta["target_scaler"]
    return (np.asarray(values).reshape(-1) - s["min_"][0]) / s["scale_"][0]


def prepare_data(raw, config, root):
    if config["HORIZON"] != 1:
        raise ValueError("Notebook này chỉ hỗ trợ dự báo một phiên tiếp theo (HORIZON=1).")
    ratios = [config["TRAIN_RATIO"], config["VAL_RATIO"], config["TEST_RATIO"]]
    if any(r <= 0 for r in ratios) or not np.isclose(sum(ratios), 1):
        raise ValueError("Tỷ lệ chia tập phải dương và tổng bằng 1.")
    frame = add_features(raw)
    n = len(frame)
    train_end, val_end = int(n * ratios[0]), int(n * sum(ratios[:2]))
    lookback = config["LOOKBACK"]
    if not (0 < lookback < train_end < val_end < n):
        raise ValueError("Không đủ dữ liệu cho lookback và ba tập; tăng khoảng ngày hoặc giảm LOOKBACK.")
    fs, ts = MinMaxScaler(), MinMaxScaler()
    fs.fit(frame.iloc[:train_end])
    ts.fit(frame.iloc[:train_end][["Close"]])
    features = fs.transform(frame).astype(np.float32)
    target = ts.transform(frame[["Close"]]).ravel().astype(np.float32)
    target_rows = np.arange(lookback, n)
    X = np.stack([features[i-lookback:i] for i in target_rows])
    y = target[target_rows]
    masks = {"train": target_rows < train_end,
             "val": (target_rows >= train_end) & (target_rows < val_end),
             "test": target_rows >= val_end}
    data, split_rows = {}, []
    for split, mask in masks.items():
        rows = target_rows[mask]
        data[f"X_{split}"] = X[mask]
        data[f"y_{split}"] = y[mask]
        data[f"actual_{split}"] = frame.Close.to_numpy()[rows]
        data[f"previous_{split}"] = frame.Close.to_numpy()[rows-1]
        data[f"dates_{split}"] = frame.index[rows].strftime("%Y-%m-%d").to_numpy(dtype=str)
        split_rows.append({"split": split, "samples": len(rows),
                           "first_target": data[f"dates_{split}"][0],
                           "last_target": data[f"dates_{split}"][-1]})
    meta = {"feature_columns": list(frame.columns), "input_dim": len(frame.columns),
            "lookback": lookback, "horizon": 1, "target": "Close",
            "feature_scaler": scaler_state(fs), "target_scaler": scaler_state(ts),
            "train_end_row": train_end, "val_end_row": val_end,
            "dropped_warmup_rows": len(raw)-len(frame), "splits": split_rows}
    assert data["dates_train"][-1] < data["dates_val"][0] < data["dates_test"][0]
    for key, value in data.items():
        if not key.startswith("dates"):
            assert np.isfinite(value).all(), key
    np.savez_compressed(root / "prepared_data.npz", **data)
    save_json(root / "preprocessing.json", meta)
    frame.to_csv(root / "features.csv")
    pd.DataFrame(split_rows).to_csv(root / "splits.csv", index=False)
    print(pd.DataFrame(split_rows).to_string(index=False))
    print("Input shape:", data["X_train"].shape, "— scaler fit trên train; val/test giữ context quá khứ.")
    return data, meta


def wavelet_inputs(data, meta, config):
    import pywt
    result = dict(data)
    close_index = meta["feature_columns"].index("Close")
    name, level = config["WAVELET"], config["WAVELET_LEVEL"]
    maximum = pywt.dwt_max_level(meta["lookback"], pywt.Wavelet(name).dec_len)
    if not 1 <= level <= maximum:
        raise ValueError(f"WAVELET_LEVEL cần trong [1, {maximum}] cho lookback hiện tại.")
    for split in ["train", "val", "test"]:
        windows = data[f"X_{split}"].copy()
        for window in windows:
            series = window[:, close_index]
            coeffs = pywt.wavedec(series, name, level=level, mode="symmetric")
            sigma = np.median(np.abs(coeffs[-1] - np.median(coeffs[-1]))) / 0.6745
            threshold = sigma * np.sqrt(2 * np.log(len(series)))
            if threshold > 0:
                coeffs[1:] = [pywt.threshold(c, threshold, mode="soft") for c in coeffs[1:]]
            window[:, close_index] = pywt.waverec(coeffs, name, mode="symmetric")[:len(series)]
        assert np.isfinite(windows).all()
        result[f"X_{split}"] = windows
    # Target thật và các đặc trưng khác giữ nguyên; không lọc toàn chuỗi trước khi split.
    return result


## Kiến trúc dùng để khôi phục checkpoint

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x):
        outputs, _ = self.lstm(x)
        return self.head(outputs[:, -1]).squeeze(-1)


def make_loader(X, y, batch_size, shuffle, seed=42, cuda=False):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
                      batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=cuda, generator=generator)


def predict_scaled(model, X, device, batch_size=256):
    model.eval()
    values = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(X[start:start+batch_size]).to(device)
            values.append(model(batch).cpu().numpy())
    return np.concatenate(values)


## Hàm đánh giá và backtest

In [ ]:
def regression_metrics(actual, predicted, previous):
    actual, predicted, previous = map(lambda x: np.asarray(x, dtype=float), [actual, predicted, previous])
    if not (actual.shape == predicted.shape == previous.shape) or actual.ndim != 1:
        raise ValueError("actual, predicted, previous phải là vector cùng shape.")
    if not all(np.isfinite(x).all() for x in [actual, predicted, previous]):
        raise ValueError("Dự đoán chứa NaN/Inf.")
    error = actual - predicted
    denom = np.sum((actual - actual.mean()) ** 2)
    return {"RMSE": float(np.sqrt(np.mean(error ** 2))), "MAE": float(np.mean(np.abs(error))),
            "MAPE_percent": float(np.mean(np.abs(error / actual)) * 100),
            "R2": float(1 - np.sum(error ** 2) / denom) if denom > 0 else None,
            "DA_percent": float(np.mean(np.sign(actual - previous) == np.sign(predicted - previous)) * 100)}


def load_model(path, device):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model = StockLSTM(**checkpoint["model_args"]).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model


def evaluate_experiments(experiments, variants, meta, source_root, output, device):
    output.mkdir(parents=True, exist_ok=True)
    rows, predictions = [], []
    raw = variants["raw"]
    naive = raw["previous_test"]
    rows.append({"run_id": "Naive", "method": "Naive", "seed": -1,
                 **regression_metrics(raw["actual_test"], naive, naive)})
    predictions.append(pd.DataFrame({"date": raw["dates_test"], "actual": raw["actual_test"],
                                    "previous": naive, "predicted": naive, "run_id": "Naive"}))
    for record in experiments:
        arrays = variants[record["variant"]]
        model = load_model(source_root / record["checkpoint"], device)
        pred = inverse_target(predict_scaled(model, arrays["X_test"], device), meta)
        rows.append({"run_id": record["run_id"], "method": record["method"], "seed": record["seed"],
                     **regression_metrics(arrays["actual_test"], pred, arrays["previous_test"]),
                     "train_seconds": record["train_seconds"],
                     "search_seconds_shared": record["search_seconds_shared"]})
        predictions.append(pd.DataFrame({"date": arrays["dates_test"], "actual": arrays["actual_test"],
                                         "previous": arrays["previous_test"], "predicted": pred,
                                         "run_id": record["run_id"]}))
        del model
    metrics = pd.DataFrame(rows)
    pred_frame = pd.concat(predictions, ignore_index=True)
    metrics.to_csv(output / "metrics_by_seed.csv", index=False)
    pred_frame.to_csv(output / "predictions.csv", index=False)
    columns = ["RMSE", "MAE", "MAPE_percent", "R2", "DA_percent"]
    summary = metrics.groupby("method", sort=False)[columns].agg(["mean", "std"])
    summary.to_csv(output / "metrics_summary.csv")
    print(metrics.to_string(index=False))
    print("\nMean/std qua seeds (std trống nếu chỉ có một lần chạy):\n", summary)
    return metrics, pred_frame


def plot_results(metrics, predictions, output):
    fig, axes = plt.subplots(2, 1, figsize=(13, 9))
    first = predictions[predictions.run_id == "Naive"]
    axes[0].plot(pd.to_datetime(first.date), first.actual, label="Actual", color="black")
    # Một seed cố định đầu tiên cho mỗi method; không chọn seed theo điểm test.
    for method, group in metrics[metrics.method != "Naive"].groupby("method", sort=False):
        run_id = group.iloc[0].run_id
        subset = predictions[predictions.run_id == run_id]
        axes[0].plot(pd.to_datetime(subset.date), subset.predicted, label=run_id, alpha=.8)
    axes[0].set_title("Giá thực tế và dự đoán trên test — seed đầu tiên")
    axes[0].legend()
    means = metrics.groupby("method", sort=False).RMSE.mean()
    means.plot.bar(ax=axes[1], rot=15, title="Test RMSE trung bình qua seeds")
    fig.tight_layout()
    fig.savefig(output / "comparison.png", dpi=160)
    plt.show()
    plt.close(fig)


def trading_metrics(actual, previous, predicted, fee, buy_hold=False):
    position = np.ones(len(actual)) if buy_hold else (predicted > previous).astype(float)
    turnover = np.abs(np.diff(np.r_[0, position]))
    returns = position * (actual / previous - 1) - fee * turnover
    returns[-1] -= fee * position[-1]  # Thanh lý vị thế cuối kỳ.
    equity = np.r_[1.0, np.cumprod(1 + returns)]
    drawdown = equity / np.maximum.accumulate(equity) - 1
    std = returns.std()
    return {"Return_percent": float((equity[-1] - 1) * 100),
            "Sharpe": float(np.sqrt(252) * returns.mean() / std) if std > 0 else 0.0,
            "Max_drawdown_percent": float(drawdown.min() * 100),
            "Turnover_units": float(turnover.sum() + position[-1])}, equity


def backtest_predictions(predictions, fee, output):
    rows = []
    fig, ax = plt.subplots(figsize=(12, 5))
    for run_id, group in predictions.groupby("run_id", sort=False):
        stats, equity = trading_metrics(group.actual.to_numpy(), group.previous.to_numpy(),
                                        group.predicted.to_numpy(), fee)
        rows.append({"run_id": run_id, **stats})
        ax.plot(equity, label=run_id, alpha=.7)
    first = predictions[predictions.run_id == "Naive"]
    stats, equity = trading_metrics(first.actual.to_numpy(), first.previous.to_numpy(),
                                    first.predicted.to_numpy(), fee, buy_hold=True)
    rows.append({"run_id": "Buy_and_hold", **stats})
    ax.plot(equity, label="Buy_and_hold", color="black", linewidth=2)
    ax.set(title="Backtest minh họa close-to-close, có phí", xlabel="Phiên test", ylabel="Vốn / vốn ban đầu")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output / "backtest.png", dpi=160)
    plt.show()
    plt.close(fig)
    table = pd.DataFrame(rows)
    table.to_csv(output / "backtest.csv", index=False)
    print(table.to_string(index=False))
    return table


In [ ]:
if not RUN_PATH:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("experiments.json")]
    if len(candidates) != 1:
        raise ValueError(f"Tìm thấy {len(candidates)} run. Đặt RUN_PATH trỏ tới thư mục chứa experiments.json.")
    RUN_PATH = str(candidates[0])
SOURCE_RUN = Path(RUN_PATH)
experiments = json.loads((SOURCE_RUN / "experiments.json").read_text(encoding="utf-8"))
meta = json.loads((SOURCE_RUN / "preprocessing.json").read_text(encoding="utf-8"))
variants = {}
for variant in {"raw"} | {record["variant"] for record in experiments}:
    name = "prepared_data.npz" if variant == "raw" else "prepared_wavelet.npz"
    with np.load(SOURCE_RUN / name, allow_pickle=False) as archive:
        variants[variant] = {key: archive[key] for key in archive.files}
DEVICE = select_device(False)
OUT = Path(EVAL_OUTPUT_ROOT) / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT.mkdir(parents=True)
save_json(OUT / "evaluation_source.json", {"run_path": str(SOURCE_RUN), "commission": COMMISSION})
metrics, predictions = evaluate_experiments(experiments, variants, meta, SOURCE_RUN, OUT, DEVICE)
plot_results(metrics, predictions, OUT)


### Backtest minh họa
Long khi dự đoán Close kế tiếp cao hơn Close hiện tại, ngược lại giữ tiền mặt.
Tính phí khi đổi vị thế, vào lệnh đầu kỳ và thanh lý cuối kỳ; drawdown có tính vốn ban đầu.
Đây là mô phỏng close-to-close giả định khớp tại Close đã quan sát, chưa mô hình hóa độ trễ,
spread, slippage hay việc chỉ biết giá đóng cửa sau khi phiên kết thúc. Giá điều chỉnh và chỉ số
`^GSPC` phục vụ nghiên cứu dự báo, không đại diện một danh mục có thể khớp lệnh trực tiếp.
Không dùng bảng lợi nhuận này làm bằng chứng hệ thống sẵn sàng giao dịch.

In [ ]:
backtest_table = backtest_predictions(predictions, COMMISSION, OUT)
display(backtest_table)
print("Saved evaluation:", OUT)


## Đọc bảng so sánh/ablation
`metrics_summary.csv` chứa trung bình và độ lệch chuẩn qua các seed.
Nếu run gốc bật Wavelet, so sánh LSTM ↔ Wavelet–LSTM để quan sát ảnh hưởng Wavelet,
và LSTM ↔ LSTM–PSO để quan sát ảnh hưởng tuning; báo cáo cả bốn cấu hình và chi phí tìm kiếm.
Nếu run gốc chỉ có LSTM, notebook này không tự sinh kết quả hybrid.
Không chọn lại siêu tham số từ bảng test. Muốn đánh giá độ bền, thiết kế thêm các khoảng thời gian/ticker
hoặc walk-forward với tập test mới trước khi chạy thí nghiệm tiếp theo.